In [0]:
def get_environment():
    try:
        env = dbutils.widgets.get("env")
        print(f"Environment is set to {env}")
        return env
    except:
        env = "dev"
        print(f"Environment is set to {env} by default")
        return env


In [0]:
import os
from pathlib import Path
from pyspark.sql import functions as F
import json

def get_config_params(config_path:str = 'src/configs/config.json'):
    cwd = Path.cwd()

    for base_path in [cwd, *cwd.parents]:
        config_file = base_path / config_path

        if config_file.exists():
            print(f"Config cargada: {config_file}")
            with open(config_file, mode="r") as f:
                return json.load(f)

    raise FileNotFoundError(
        f"No se encontró {config_path} buscando desde {cwd} hacia arriba"
    )

In [0]:
def silver_init_audit():
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {licenses_catalog}.audit.audit_licenses (
      table_orig STRING,
      table_dest STRING,
      last_cdf_version_ingested INTEGER,
      ts_audit TIMESTAMP)""")

In [0]:
from pyspark.sql.functions import max, col

def get_last_bronze_version(bronze_full_tablename:str) -> int:

    last_bronze_version = spark.sql(f"""DESCRIBE HISTORY {bronze_full_tablename}""")\
        .where(
            (col("operation") == "WRITE")|
            (col("operation").like("CREATE TABLE%")))\
        .agg(max(col("version")).alias("max_version"))\
        .first()["max_version"] 
    print(f"last_bronze_version encountered: {last_bronze_version}")

    return last_bronze_version

In [0]:
def decide_source_to_load(catalog:str, bronze_full_tablename:str, last_bronze_version:int):
    last_cdf_version_ingested = spark.sql(f"""
                                        SELECT MAX(last_cdf_version_ingested) as last_cdf_version_ingested
                                        FROM {licenses_catalog}.audit.audit_licenses
                                        """).first()["last_cdf_version_ingested"]
    
    if last_cdf_version_ingested is not None and last_bronze_version > last_cdf_version_ingested:
        reference_cdf_version = last_cdf_version_ingested + 1
        source_to_load = spark.sql(f"""
                        SELECT * FROM table_changes({bronze_full_tablename }, {reference_cdf_version})
                        """)
        return source_to_load
        print(f"source to load set to incremental")
    elif last_cdf_version_ingested is None:
        source_to_load = spark.sql(f"""
                        SELECT * FROM {bronze_full_tablename}
                        """)
        print(f"source is set to full bronze")
        return source_to_load
    else:
        print("No new data to load")
    
